# LSScore unified final conference analysis

This is the same analysis pipeline for CLASSE, ASAP, and PERSUADE 2.0.
Change only `DATASET_NAME` in the configuration cell.

The pipeline:

1. reuses parameter-independent pair features from an old locked-test checkpoint;
2. exhaustively evaluates the complete 3,584-configuration validation grid;
3. locks the validation winner before scoring TEST;
4. reuses only approved baseline columns from old baseline checkpoints;
5. adds the candidate-length baseline and held-out length-confound analysis;
6. performs hierarchical prompt-cluster inference, Holm correction, and LOPO;
7. reports Williams tests as supplementary evidence;
8. evaluates locked-parameter ablations for the appendix.

Old HPO parameters, old LSScore values, excluded metric variants, and old
row-level bootstrap results are never reused.

In [ ]:
# Colab setup and imports
from __future__ import annotations

import hashlib
import json
import math
import os
import pickle
import random
import re
import sys
from datetime import datetime
from itertools import product
from pathlib import Path
from typing import Any, Iterable, Sequence

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.stats import kendalltau, pearsonr, rankdata, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

try:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

## 1. Dataset selection

Change only the following value to one of:

- `"CLASSE"`
- `"ASAP"`
- `"PERSUADE"`

In [ ]:
DATASET_NAME = "ASAP"  # Change only this line.

SEED = 42
SEARCH_SEED = 2026
BOOTSTRAP_REPETITIONS = 2_000
VAL_PAIRWISE_SAMPLE = 20_000
RUN_ABLATION = True
FORCE_RERUN_FULL_GRID = False
FORCE_RERUN_LOCKED_SCORES = False
GRID_PROGRESS_EVERY = 25

random.seed(SEED)
np.random.seed(SEED)


DATASET_CONFIGS = {
    "ASAP": {
        "old_checkpoint_dir": Path(
            "/content/drive/MyDrive/Colab Notebooks/Optimized Code/"
            "checkpoints/asap"
        ),
        "expected_test_rows": 1_951,
        "expected_prompts": 8,
    },
    "CLASSE": {
        "old_checkpoint_dir": Path(
            "/content/drive/MyDrive/Colab Notebooks/LSScore/LSScore/"
            "checkpoints/classe"
        ),
        "expected_test_rows": 1_687,
        "expected_prompts": 6,
    },
    "PERSUADE": {
        "old_checkpoint_dir": Path(
            "/content/drive/MyDrive/Colab Notebooks/Optimized Code/"
            "checkpoints/persuade"
        ),
        "expected_test_rows": 2_353,
        "expected_prompts": 15,
    },
}

DATASET_NAME = DATASET_NAME.upper().strip()
if DATASET_NAME not in DATASET_CONFIGS:
    raise ValueError(
        f"DATASET_NAME must be one of {list(DATASET_CONFIGS)}, "
        f"not {DATASET_NAME!r}."
    )

DATASET_CONFIG = DATASET_CONFIGS[DATASET_NAME]
OLD_CHECKPOINT_DIR = DATASET_CONFIG["old_checkpoint_dir"]

# New results never overwrite the old checkpoints.
NEW_EXPERIMENT_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/LSScore/LSScore/"
    "conference_fullgrid_cluster_length_v1"
)
NEW_CHECKPOINT_DIR = NEW_EXPERIMENT_ROOT / "checkpoints" / DATASET_NAME.lower()
OUTPUT_DIR = NEW_EXPERIMENT_ROOT / "outputs" / DATASET_NAME.lower()
NEW_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_NAME)
print("Old reusable checkpoints:", OLD_CHECKPOINT_DIR)
print("New checkpoints:", NEW_CHECKPOINT_DIR)
print("Outputs:", OUTPUT_DIR)

## 2. Search space

The grid is identical for all three datasets and contains exactly 3,584
configurations. It is evaluated exhaustively on VALIDATION.

In [ ]:
N_LIST = [1, 2, 3]

TAU_VALUES = [
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
]

DELTA_VALUES = [
    0.60,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
]

ALPHA_VALUES = [
    (0.50, 0.35, 0.15),
    (0.45, 0.40, 0.15),
    (0.40, 0.40, 0.20),
    (0.35, 0.45, 0.20),
    (0.30, 0.50, 0.20),
    (0.25, 0.55, 0.20),
    (0.20, 0.60, 0.20),
    (0.20, 0.55, 0.25),
]

BETA_VALUES = [
    0.50,
    0.75,
    1.00,
    2.00,
]

USE_BP_VALUES = [
    True,
    False,
]

FULL_SEARCH_SPACE = list(
    product(
        TAU_VALUES,
        DELTA_VALUES,
        ALPHA_VALUES,
        BETA_VALUES,
        USE_BP_VALUES,
    )
)

assert len(FULL_SEARCH_SPACE) == 3_584

SEARCH_SPECIFICATION = {
    "dataset": DATASET_NAME,
    "tau": TAU_VALUES,
    "delta": DELTA_VALUES,
    "alpha": ALPHA_VALUES,
    "beta": BETA_VALUES,
    "use_bp": USE_BP_VALUES,
    "selection": [
        "val_avg_corr_desc",
        "val_pairwise_accuracy_desc",
        "val_calibrated_rmse_asc",
        "val_calibrated_mae_asc",
    ],
}

SEARCH_SPEC_SIGNATURE = hashlib.sha256(
    json.dumps(
        SEARCH_SPECIFICATION,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:16]

print("Full validation grid:", len(FULL_SEARCH_SPACE))
print("Search-specification signature:", SEARCH_SPEC_SIGNATURE)

## 3. Checkpoint helpers

The old checkpoints are treated as feature/baseline caches, not as final
scientific results.

| Old checkpoint | What this notebook may reuse |
|---|---|
| `06_locked_test_complete.pkl` | Split membership, text/IDs, normalised labels, and parameter-independent row features |
| `06_local800_locked_test_complete.pkl` | PERSUADE fallback for the same core objects only |
| `07_standard_baselines_complete.pkl` | BLEU, ROUGE, and BERTScore columns only |
| `07_after_seascore.pkl` | SEAScore column only |
| `07_after_holms.pkl` | HOLMS column only |
| `07_after_smile.pkl` | SMILe arithmetic- and harmonic-mean columns only |
| `07_external_baselines_complete.pkl` | Approved external ROUGE-hybrid column only |

Do **not** reuse old HPO/stability checkpoints, old selected parameters, old
LSScore columns, old significance/bootstrap tables, or old final result files.
This notebook writes a separate, versioned checkpoint directory and never
overwrites the old files.

In [ ]:
def checkpoint_path(directory: Path, name: str) -> Path:
    return directory / f"{name}.pkl"


def read_checkpoint_file(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")

    with path.open("rb") as handle:
        payload = pickle.load(handle)

    if not isinstance(payload, dict) or "variables" not in payload:
        raise ValueError(f"Invalid checkpoint format: {path}")

    stored_dataset = str(payload.get("dataset", "")).upper()
    if stored_dataset and stored_dataset != DATASET_NAME:
        raise ValueError(
            f"Checkpoint dataset mismatch: expected {DATASET_NAME}, "
            f"found {stored_dataset} in {path}."
        )

    return payload


def read_old_checkpoint(name: str) -> dict[str, Any]:
    return read_checkpoint_file(
        checkpoint_path(OLD_CHECKPOINT_DIR, name)
    )


def read_new_checkpoint(name: str) -> dict[str, Any]:
    return read_checkpoint_file(
        checkpoint_path(NEW_CHECKPOINT_DIR, name)
    )


def save_new_checkpoint(
    name: str,
    variables: dict[str, Any],
    metadata: dict[str, Any] | None = None,
) -> Path:
    path = checkpoint_path(NEW_CHECKPOINT_DIR, name)
    temporary_path = path.with_suffix(".pkl.tmp")

    payload = {
        "checkpoint": name,
        "dataset": DATASET_NAME,
        "experiment": "conference_fullgrid_cluster_length_v1",
        "search_signature": SEARCH_SPEC_SIGNATURE,
        "data_fingerprint": DATA_FINGERPRINT,
        "run_signature": RUN_SIGNATURE,
        "saved_at": datetime.now().isoformat(timespec="seconds"),
        "metadata": metadata or {},
        "variables": variables,
    }

    with temporary_path.open("wb") as handle:
        pickle.dump(
            payload,
            handle,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    os.replace(temporary_path, path)
    print(f"Saved new checkpoint: {path}")
    return path


def validate_new_checkpoint(payload: dict[str, Any]) -> None:
    if payload.get("search_signature") != SEARCH_SPEC_SIGNATURE:
        raise ValueError(
            "The new checkpoint was created with a different search "
            "specification. Set the relevant FORCE_RERUN flag to True."
        )
    if payload.get("data_fingerprint") != DATA_FINGERPRINT:
        raise ValueError(
            "The reusable split/features changed after this checkpoint was "
            "created. Use a new experiment directory or set the relevant "
            "FORCE_RERUN flag to True."
        )

## 4. Reuse the parameter-independent split and LSScore features

`06_locked_test_complete` is reusable only for:

- TRAIN/VALIDATION/TEST membership;
- normalised human labels;
- text and item identifiers;
- parameter-independent `train_row_data`, `val_row_data`, and `test_row_data`.

The old `best_params` and old `ls_score` columns are deliberately discarded.

In [ ]:
CORE_CHECKPOINT_CANDIDATES = [
    "06_locked_test_complete",
    "06_local800_locked_test_complete",
]


def load_reusable_core() -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    list[dict[str, Any]],
    list[dict[str, Any]],
    list[dict[str, Any]],
]:
    errors = []

    for name in CORE_CHECKPOINT_CANDIDATES:
        try:
            payload = read_old_checkpoint(name)
        except Exception as exc:
            errors.append(f"{name}: {exc}")
            continue

        variables = payload["variables"]
        required = {
            "train_df",
            "val_df",
            "test_df",
            "train_row_data",
            "val_row_data",
            "test_row_data",
        }

        if not required.issubset(variables):
            missing = sorted(required - set(variables))
            errors.append(f"{name}: missing {missing}")
            continue

        print(f"Reusing parameter-independent data from: {name}")

        train = variables["train_df"].copy().reset_index(drop=True)
        validation = variables["val_df"].copy().reset_index(drop=True)
        test = variables["test_df"].copy().reset_index(drop=True)

        # Never reuse old LSScore values or the removed length analysis.
        for frame in [train, validation, test]:
            frame.drop(
                columns=["ls_score", "candidate_length"],
                errors="ignore",
                inplace=True,
            )

        return (
            train,
            validation,
            test,
            variables["train_row_data"],
            variables["val_row_data"],
            variables["test_row_data"],
        )

    raise RuntimeError(
        "No reusable locked-test checkpoint was found.\n"
        + "\n".join(errors)
    )


(
    train_df,
    val_df,
    test_df,
    train_row_data,
    val_row_data,
    test_row_data,
) = load_reusable_core()

assert len(train_df) == len(train_row_data)
assert len(val_df) == len(val_row_data)
assert len(test_df) == len(test_row_data)
assert len(test_df) == DATASET_CONFIG["expected_test_rows"]
assert test_df["prompt_id"].nunique() == DATASET_CONFIG["expected_prompts"]

KEY_COLUMNS = [
    "prompt_id",
    "ref_item_id",
    "cand_item_id",
]


def split_data_fingerprint(frames: Sequence[pd.DataFrame]) -> str:
    digest = hashlib.sha256()
    fingerprint_columns = KEY_COLUMNS + ["normalised_human_score"]
    for frame in frames:
        missing = [
            column
            for column in fingerprint_columns
            if column not in frame.columns
        ]
        if missing:
            raise KeyError(
                f"Cannot fingerprint split; missing columns: {missing}"
            )
        hashed = pd.util.hash_pandas_object(
            frame[fingerprint_columns].astype(str),
            index=False,
        ).to_numpy(dtype=np.uint64)
        digest.update(hashed.tobytes())
    return digest.hexdigest()[:16]


DATA_FINGERPRINT = split_data_fingerprint([train_df, val_df, test_df])
RUN_SIGNATURE = hashlib.sha256(
    f"{SEARCH_SPEC_SIGNATURE}:{DATA_FINGERPRINT}".encode("utf-8")
).hexdigest()[:16]

print(
    f"Rows: TRAIN={len(train_df):,}, "
    f"VALIDATION={len(val_df):,}, TEST={len(test_df):,}"
)
print("TEST prompts:", test_df["prompt_id"].nunique())
print("Data fingerprint:", DATA_FINGERPRINT)
print("Run signature:", RUN_SIGNATURE)

## 5. Split and pseudo-reference safety checks

In [ ]:
for frame_name, frame in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df),
]:
    missing = [column for column in KEY_COLUMNS if column not in frame.columns]
    if missing:
        raise KeyError(f"{frame_name} is missing key columns: {missing}")

    if frame.duplicated(KEY_COLUMNS).any():
        raise ValueError(f"{frame_name} contains duplicate pair keys.")

train_candidates = set(train_df["cand_item_id"].astype(str))
val_candidates = set(val_df["cand_item_id"].astype(str))
test_candidates = set(test_df["cand_item_id"].astype(str))

assert train_candidates.isdisjoint(val_candidates)
assert train_candidates.isdisjoint(test_candidates)
assert val_candidates.isdisjoint(test_candidates)

train_reference_ids = set(train_df["ref_item_id"].astype(str))
assert train_reference_ids.isdisjoint(val_candidates)
assert train_reference_ids.isdisjoint(test_candidates)

for frame_name, frame in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df),
]:
    assert frame["normalised_human_score"].notna().all()
    assert frame["prompt_id"].notna().all()

print("Split, item-ID, and pseudo-reference safety checks passed.")

## 6. LSScore scoring functions

In [ ]:
def get_match_values(
    exact: np.ndarray,
    similarity: np.ndarray,
    tau: float,
    delta: float,
    mode: str = "joint",
) -> np.ndarray:
    exact = np.asarray(exact, dtype=bool)
    similarity = np.asarray(similarity, dtype=float)

    if mode == "joint":
        return np.where(
            exact,
            1.0,
            np.where(
                (~exact) & (similarity >= tau),
                delta * similarity,
                0.0,
            ),
        )

    if mode == "lexical":
        return exact.astype(np.float32)

    if mode == "semantic":
        return np.where(
            (~exact) & (similarity >= tau),
            delta * similarity,
            0.0,
        )

    raise ValueError("mode must be 'joint', 'lexical', or 'semantic'.")


def fast_ls_score(
    row_data: dict[str, Any],
    n_list: Sequence[int],
    alpha: Sequence[float],
    tau: float,
    delta: float,
    beta: float,
    use_bp: bool = True,
    mode: str = "joint",
) -> float:
    base_score = 0.0

    for n, weight in zip(n_list, alpha):
        level = row_data["per_n"].get(n)
        if level is None:
            continue

        recall_match = get_match_values(
            level["exact_recall"],
            level["best_sim_recall"],
            tau,
            delta,
            mode=mode,
        )
        precision_match = get_match_values(
            level["exact_prec"],
            level["best_sim_prec"],
            tau,
            delta,
            mode=mode,
        )

        recall_denominator = float(level["idf_recall"].sum())
        precision_denominator = float(level["idf_prec"].sum())

        recall = (
            float(
                (level["idf_recall"] * recall_match).sum()
                / recall_denominator
            )
            if recall_denominator > 0
            else 0.0
        )
        precision = (
            float(
                (level["idf_prec"] * precision_match).sum()
                / precision_denominator
            )
            if precision_denominator > 0
            else 0.0
        )

        denominator = beta**2 * precision + recall
        f_beta = (
            (1 + beta**2) * precision * recall / denominator
            if denominator > 0
            else 0.0
        )
        base_score += weight * f_beta

    if not use_bp:
        return float(base_score)

    reference_length = row_data["ref_len"]
    candidate_length = row_data["cand_len"]

    brevity_penalty = (
        1.0
        if candidate_length >= reference_length
        else np.exp(
            1 - reference_length / (candidate_length + 1e-10)
        )
    )
    return float(brevity_penalty * base_score)


def score_rows(
    row_data: Sequence[dict[str, Any]],
    params: dict[str, Any],
    *,
    mode: str = "joint",
    use_bp_override: bool | None = None,
) -> np.ndarray:
    use_bp = (
        params["use_bp"]
        if use_bp_override is None
        else bool(use_bp_override)
    )

    return np.fromiter(
        (
            fast_ls_score(
                record,
                N_LIST,
                params["alpha"],
                params["tau"],
                params["delta"],
                params["beta"],
                use_bp=use_bp,
                mode=mode,
            )
            for record in row_data
        ),
        dtype=np.float64,
        count=len(row_data),
    )

## 7. General evaluation helpers

In [ ]:
def safe_corr(
    x: Sequence[float],
    y: Sequence[float],
    correlation_function,
) -> float:
    x_array = np.asarray(x, dtype=float)
    y_array = np.asarray(y, dtype=float)

    valid = np.isfinite(x_array) & np.isfinite(y_array)
    x_array = x_array[valid]
    y_array = y_array[valid]

    if (
        len(x_array) < 3
        or np.unique(x_array).size <= 1
        or np.unique(y_array).size <= 1
    ):
        return np.nan

    return float(correlation_function(x_array, y_array)[0])


def calibrated_errors_from_arrays(
    train_scores: Sequence[float],
    train_human: Sequence[float],
    evaluation_scores: Sequence[float],
    evaluation_human: Sequence[float],
) -> tuple[float, float]:
    model = LinearRegression()
    model.fit(
        np.asarray(train_scores, dtype=float).reshape(-1, 1),
        np.asarray(train_human, dtype=float),
    )
    predictions = model.predict(
        np.asarray(evaluation_scores, dtype=float).reshape(-1, 1)
    )
    target = np.asarray(evaluation_human, dtype=float)
    return (
        float(np.sqrt(mean_squared_error(target, predictions))),
        float(mean_absolute_error(target, predictions)),
    )


def prepare_pairwise_validation_sample(
    df: pd.DataFrame,
    max_pairs: int = 20_000,
    seed: int = 2026,
    human_col: str = "normalised_human_score",
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    sampled_i: list[int] = []
    sampled_j: list[int] = []
    human_sign: list[float] = []

    groups = list(df.groupby("prompt_id", sort=False))
    per_group_target = max(
        1,
        int(np.ceil(max_pairs / max(1, len(groups)))),
    )

    for _, group in groups:
        positions = group.index.to_numpy()
        if len(positions) < 2:
            continue

        human = df.loc[positions, human_col].to_numpy(dtype=float)
        collected = 0
        attempts = 0

        while collected < per_group_target and attempts < per_group_target * 30:
            first, second = rng.choice(
                len(positions),
                size=2,
                replace=False,
            )
            attempts += 1
            difference = human[first] - human[second]
            if np.isclose(difference, 0.0):
                continue

            sampled_i.append(int(positions[first]))
            sampled_j.append(int(positions[second]))
            human_sign.append(float(np.sign(difference)))
            collected += 1

    if len(sampled_i) > max_pairs:
        keep = rng.choice(
            len(sampled_i),
            size=max_pairs,
            replace=False,
        )
        sampled_i = np.asarray(sampled_i)[keep].tolist()
        sampled_j = np.asarray(sampled_j)[keep].tolist()
        human_sign = np.asarray(human_sign)[keep].tolist()

    return (
        np.asarray(sampled_i, dtype=int),
        np.asarray(sampled_j, dtype=int),
        np.asarray(human_sign, dtype=float),
    )


def pairwise_accuracy_from_sample(
    scores: Sequence[float],
    pair_sample: tuple[np.ndarray, np.ndarray, np.ndarray],
    tie_credit: float = 0.5,
) -> float:
    first, second, human_sign = pair_sample
    score_array = np.asarray(scores, dtype=float)
    metric_difference = score_array[first] - score_array[second]
    metric_ties = np.isclose(metric_difference, 0.0)
    correct = np.sign(metric_difference) == human_sign
    credit = np.where(metric_ties, tie_credit, correct.astype(float))
    return float(np.mean(credit)) if len(credit) else np.nan


def exact_pairwise_accuracy(
    df: pd.DataFrame,
    metric_col: str,
    human_col: str = "normalised_human_score",
    tie_credit: float = 0.5,
) -> float:
    correct = 0.0
    total = 0

    for _, group in df.groupby("prompt_id", sort=False):
        values = group[[metric_col, human_col]].to_numpy(dtype=float)
        for first in range(len(values)):
            for second in range(first + 1, len(values)):
                metric_difference = values[first, 0] - values[second, 0]
                human_difference = values[first, 1] - values[second, 1]
                if np.isclose(human_difference, 0.0):
                    continue

                total += 1
                if np.isclose(metric_difference, 0.0):
                    correct += tie_credit
                elif np.sign(metric_difference) == np.sign(human_difference):
                    correct += 1.0

    return correct / total if total else np.nan


def average_correlation(
    scores: Sequence[float],
    human: Sequence[float],
) -> float:
    pearson = safe_corr(scores, human, pearsonr)
    spearman = safe_corr(scores, human, spearmanr)
    if not np.isfinite(pearson) or not np.isfinite(spearman):
        return np.nan
    return float((pearson + spearman) / 2.0)

## 8. Exhaustive validation search

TEST labels are not used anywhere in this section.

In [ ]:
FULL_GRID_CHECKPOINT = "05_fullgrid_validation_complete"
FULL_GRID_PROGRESS_CHECKPOINT = "05_fullgrid_validation_progress"
full_grid_path = checkpoint_path(NEW_CHECKPOINT_DIR, FULL_GRID_CHECKPOINT)
full_grid_progress_path = checkpoint_path(
    NEW_CHECKPOINT_DIR,
    FULL_GRID_PROGRESS_CHECKPOINT,
)

if full_grid_path.exists() and not FORCE_RERUN_FULL_GRID:
    full_grid_payload = read_new_checkpoint(FULL_GRID_CHECKPOINT)
    validate_new_checkpoint(full_grid_payload)
    grid_df = full_grid_payload["variables"]["grid_df"]
    best_params = full_grid_payload["variables"]["best_params"]
    print("Loaded compatible exhaustive-grid result.")
else:
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    train_human = train_df["normalised_human_score"].to_numpy(dtype=float)
    val_human = val_df["normalised_human_score"].to_numpy(dtype=float)

    validation_pair_sample = prepare_pairwise_validation_sample(
        val_df,
        max_pairs=VAL_PAIRWISE_SAMPLE,
        seed=SEARCH_SEED,
    )

    grid_rows: list[dict[str, Any]] = []
    start_index = 0

    if full_grid_progress_path.exists() and not FORCE_RERUN_FULL_GRID:
        progress_payload = read_new_checkpoint(
            FULL_GRID_PROGRESS_CHECKPOINT
        )
        validate_new_checkpoint(progress_payload)
        progress_variables = progress_payload["variables"]
        grid_rows = list(progress_variables["grid_rows"])
        start_index = int(progress_variables["next_index"])
        if not 0 <= start_index <= len(FULL_SEARCH_SPACE):
            raise ValueError("Invalid exhaustive-grid progress index.")
        print(
            f"Resuming exhaustive grid at {start_index:,}/"
            f"{len(FULL_SEARCH_SPACE):,}."
        )

    remaining_grid = FULL_SEARCH_SPACE[start_index:]
    for configuration_index, (
        tau,
        delta,
        alpha,
        beta,
        use_bp,
    ) in enumerate(
        tqdm(
            remaining_grid,
            desc=f"{DATASET_NAME}: exhaustive VALIDATION grid",
            initial=start_index,
            total=len(FULL_SEARCH_SPACE),
        ),
        start=start_index,
    ):
        params = {
            "tau": float(tau),
            "delta": float(delta),
            "alpha": list(alpha),
            "beta": float(beta),
            "use_bp": bool(use_bp),
        }

        train_scores = score_rows(train_row_data, params)
        validation_scores = score_rows(val_row_data, params)

        pearson = safe_corr(validation_scores, val_human, pearsonr)
        spearman = safe_corr(validation_scores, val_human, spearmanr)
        kendall = safe_corr(validation_scores, val_human, kendalltau)

        valid_correlations = all(
            np.isfinite(value)
            for value in [pearson, spearman, kendall]
        )

        if valid_correlations:
            rmse, mae = calibrated_errors_from_arrays(
                train_scores,
                train_human,
                validation_scores,
                val_human,
            )
            pairwise = pairwise_accuracy_from_sample(
                validation_scores,
                validation_pair_sample,
                tie_credit=0.5,
            )

            grid_rows.append(
                {
                    **params,
                    "alpha": tuple(alpha),
                    "val_pearson": pearson,
                    "val_spearman": spearman,
                    "val_kendall": kendall,
                    "val_avg_corr": (pearson + spearman) / 2.0,
                    "val_pairwise_accuracy": pairwise,
                    "val_calibrated_rmse": rmse,
                    "val_calibrated_mae": mae,
                }
            )

        next_index = configuration_index + 1
        if (
            next_index % GRID_PROGRESS_EVERY == 0
            or next_index == len(FULL_SEARCH_SPACE)
        ):
            save_new_checkpoint(
                FULL_GRID_PROGRESS_CHECKPOINT,
                {
                    "grid_rows": grid_rows,
                    "next_index": next_index,
                },
                metadata={
                    "completed_configurations": next_index,
                    "total_configurations": len(FULL_SEARCH_SPACE),
                },
            )

    grid_df = pd.DataFrame(grid_rows)
    if len(grid_df) != len(FULL_SEARCH_SPACE):
        print(
            f"Warning: {len(FULL_SEARCH_SPACE) - len(grid_df)} "
            "configurations produced invalid correlations."
        )

    # Primary outcome: AvgCorr. Secondary criteria break exact or near ties.
    grid_df = (
        grid_df.sort_values(
            [
                "val_avg_corr",
                "val_pairwise_accuracy",
                "val_calibrated_rmse",
                "val_calibrated_mae",
            ],
            ascending=[False, False, True, True],
        )
        .reset_index(drop=True)
    )

    winner = grid_df.iloc[0]
    best_params = {
        "tau": float(winner["tau"]),
        "delta": float(winner["delta"]),
        "alpha": [float(value) for value in winner["alpha"]],
        "beta": float(winner["beta"]),
        "use_bp": bool(winner["use_bp"]),
    }

    save_new_checkpoint(
        FULL_GRID_CHECKPOINT,
        {
            "grid_df": grid_df,
            "best_params": best_params,
        },
        metadata={
            "selection_rule": SEARCH_SPECIFICATION["selection"],
            "n_configurations": len(FULL_SEARCH_SPACE),
        },
    )

grid_df.to_csv(
    OUTPUT_DIR / "fullgrid_validation_results.csv",
    index=False,
)

print("\nTop 20 exhaustive-grid configurations:")
display(
    grid_df[
        [
            "tau",
            "delta",
            "alpha",
            "beta",
            "use_bp",
            "val_pearson",
            "val_spearman",
            "val_kendall",
            "val_avg_corr",
            "val_pairwise_accuracy",
            "val_calibrated_rmse",
            "val_calibrated_mae",
        ]
    ].head(20).round(6)
)
print("Locked VALIDATION winner:", best_params)

## 9. Score all splits with the locked validation winner

In [ ]:
LOCKED_SCORE_CHECKPOINT = "06_fullgrid_locked_scores_complete"
locked_score_path = checkpoint_path(NEW_CHECKPOINT_DIR, LOCKED_SCORE_CHECKPOINT)

if locked_score_path.exists() and not FORCE_RERUN_LOCKED_SCORES:
    locked_payload = read_new_checkpoint(LOCKED_SCORE_CHECKPOINT)
    validate_new_checkpoint(locked_payload)
    locked_variables = locked_payload["variables"]
    train_df = locked_variables["train_df"].copy()
    val_df = locked_variables["val_df"].copy()
    test_df = locked_variables["test_df"].copy()
    stored_params = locked_variables["best_params"]
    if stored_params != best_params:
        raise ValueError("Locked-score checkpoint parameters do not match HPO.")
    print("Loaded compatible locked LSScore values.")
else:
    train_df["ls_score"] = score_rows(train_row_data, best_params)
    val_df["ls_score"] = score_rows(val_row_data, best_params)
    test_df["ls_score"] = score_rows(test_row_data, best_params)

    save_new_checkpoint(
        LOCKED_SCORE_CHECKPOINT,
        {
            "best_params": best_params,
            "train_df": train_df,
            "val_df": val_df,
            "test_df": test_df,
        },
    )

assert train_df["ls_score"].notna().all()
assert val_df["ls_score"].notna().all()
assert test_df["ls_score"].notna().all()
print("TEST was scored only after the validation winner was locked.")

## 10. Reuse approved old baseline checkpoints

Only baseline columns are imported. Old LSScore values and old parameter
dictionaries are ignored.

In [ ]:
APPROVED_BASELINE_SOURCES = {
    "bleu": [
        "07_standard_baselines_complete",
        "07_external_baselines_complete",
    ],
    "rouge": [
        "07_standard_baselines_complete",
        "07_external_baselines_complete",
    ],
    "bert_score": [
        "07_standard_baselines_complete",
        "07_external_baselines_complete",
    ],
    "sea_score": [
        "07_after_seascore",
        "07_external_baselines_complete",
    ],
    "holms": [
        "07_after_holms",
        "07_external_baselines_complete",
    ],
    "smile_avg": [
        "07_after_smile",
        "07_external_baselines_complete",
    ],
    "smile_hm": [
        "07_after_smile",
        "07_external_baselines_complete",
    ],
    "hybrideval_rouge": [
        "07_external_baselines_complete",
    ],
}


def extract_metric_frames(
    payload: dict[str, Any],
    metric: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    variables = payload["variables"]

    baseline_frame = variables.get("baseline_df")
    split_column = None
    if isinstance(baseline_frame, pd.DataFrame):
        if "_baseline_split" in baseline_frame.columns:
            split_column = "_baseline_split"
        elif "split" in baseline_frame.columns:
            split_column = "split"

    if (
        isinstance(baseline_frame, pd.DataFrame)
        and metric in baseline_frame.columns
        and split_column is not None
    ):
        split_labels = baseline_frame[split_column].astype(str).str.lower()
        old_train = baseline_frame.loc[
            split_labels == "train",
            KEY_COLUMNS + [metric],
        ].copy()
        old_test = baseline_frame.loc[
            split_labels == "test",
            KEY_COLUMNS + [metric],
        ].copy()
        return old_train, old_test

    old_train = variables.get("train_df")
    old_test = variables.get("test_df")
    if (
        isinstance(old_train, pd.DataFrame)
        and isinstance(old_test, pd.DataFrame)
        and metric in old_train.columns
        and metric in old_test.columns
    ):
        return (
            old_train[KEY_COLUMNS + [metric]].copy(),
            old_test[KEY_COLUMNS + [metric]].copy(),
        )

    raise KeyError(f"Metric {metric!r} is absent from this checkpoint.")


def load_old_metric(metric: str) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    errors = []
    for checkpoint_name in APPROVED_BASELINE_SOURCES[metric]:
        try:
            payload = read_old_checkpoint(checkpoint_name)
            old_train, old_test = extract_metric_frames(payload, metric)
            return old_train, old_test, checkpoint_name
        except Exception as exc:
            errors.append(f"{checkpoint_name}: {exc}")

    raise RuntimeError(
        f"Could not recover {metric!r}.\n" + "\n".join(errors)
    )


def merge_metric(
    current: pd.DataFrame,
    recovered: pd.DataFrame,
    metric: str,
    split_name: str,
) -> pd.DataFrame:
    recovered = recovered.copy()
    if recovered.duplicated(KEY_COLUMNS).any():
        raise ValueError(f"Recovered {metric} has duplicate {split_name} keys.")

    output = current.drop(columns=[metric], errors="ignore").merge(
        recovered,
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )

    missing = int(output[metric].isna().sum())
    if missing:
        raise ValueError(
            f"{metric}: {missing} missing values after {split_name} merge."
        )
    if len(output) != len(current):
        raise ValueError(f"{metric}: {split_name} row count changed during merge.")

    return output


baseline_reuse_rows = []
for metric in APPROVED_BASELINE_SOURCES:
    recovered_train, recovered_test, source_checkpoint = load_old_metric(metric)
    train_df = merge_metric(train_df, recovered_train, metric, "TRAIN")
    test_df = merge_metric(test_df, recovered_test, metric, "TEST")
    baseline_reuse_rows.append(
        {
            "metric": metric,
            "source_checkpoint": source_checkpoint,
            "reused": True,
        }
    )
    print(f"Reused {metric} from {source_checkpoint}")

baseline_reuse_df = pd.DataFrame(baseline_reuse_rows)
baseline_reuse_df.to_csv(
    OUTPUT_DIR / "baseline_checkpoint_reuse.csv",
    index=False,
)

## 11. Candidate-length baseline

In [ ]:
WORD_PATTERN = re.compile(
    r"\b\w+(?:['’-]\w+)*\b",
    flags=re.UNICODE,
)


def count_candidate_words(text: Any) -> int:
    return len(WORD_PATTERN.findall(str(text).lower()))


for frame_name, frame in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df),
]:
    frame["candidate_length"] = (
        frame["candidate_original"]
        .apply(count_candidate_words)
        .astype(float)
    )
    if (frame["candidate_length"] <= 0).any():
        raise ValueError(f"{frame_name} contains a zero-length candidate.")

    print(
        f"{frame_name}: mean={frame['candidate_length'].mean():.1f}, "
        f"median={frame['candidate_length'].median():.1f}, "
        f"range=[{frame['candidate_length'].min():.0f}, "
        f"{frame['candidate_length'].max():.0f}]"
    )

## 12. Final held-out metric table

In [ ]:
FINAL_METRICS = [
    "ls_score",
    "candidate_length",
    "bleu",
    "rouge",
    "bert_score",
    "sea_score",
    "holms",
    "smile_avg",
    "smile_hm",
    "hybrideval_rouge",
]

for metric in FINAL_METRICS:
    if metric not in train_df.columns or metric not in test_df.columns:
        raise KeyError(f"Final metric is incomplete: {metric}")
    if train_df[metric].isna().any() or test_df[metric].isna().any():
        raise ValueError(f"Final metric contains missing values: {metric}")


def calibrated_errors(
    metric: str,
    target_col: str = "normalised_human_score",
) -> tuple[float, float]:
    return calibrated_errors_from_arrays(
        train_df[metric],
        train_df[target_col],
        test_df[metric],
        test_df[target_col],
    )


result_rows = []
test_human = test_df["normalised_human_score"].to_numpy(dtype=float)

for metric in FINAL_METRICS:
    pearson = safe_corr(test_df[metric], test_human, pearsonr)
    spearman = safe_corr(test_df[metric], test_human, spearmanr)
    kendall = safe_corr(test_df[metric], test_human, kendalltau)
    rmse, mae = calibrated_errors(metric)

    result_rows.append(
        {
            "metric": metric,
            "pearson_r": pearson,
            "spearman_r": spearman,
            "kendall_tau": kendall,
            "avg_correlation": (pearson + spearman) / 2.0,
            "calibrated_rmse": rmse,
            "calibrated_mae": mae,
            "pairwise_accuracy": exact_pairwise_accuracy(test_df, metric),
        }
    )

results_df = pd.DataFrame(result_rows)
results_df.to_csv(OUTPUT_DIR / "final_heldout_results.csv", index=False)

print("\nFINAL HELD-OUT TEST RESULTS")
print("AvgCorr = (Pearson + Spearman) / 2.")
print("RMSE/MAE are calibrated on TRAIN only.")
display(results_df.round(6))

## 13. Length-confound analysis

In [ ]:
def partial_correlation_controlling_length_and_prompt(
    df: pd.DataFrame,
    *,
    rank_transform: bool = False,
) -> dict[str, Any]:
    columns = [
        "ls_score",
        "normalised_human_score",
        "candidate_length",
        "prompt_id",
    ]
    data = df[columns].dropna().reset_index(drop=True)

    lsscore = data["ls_score"].to_numpy(dtype=float)
    human = data["normalised_human_score"].to_numpy(dtype=float)
    length = data["candidate_length"].to_numpy(dtype=float)

    if rank_transform:
        lsscore = rankdata(lsscore)
        human = rankdata(human)
        length = rankdata(length)

    prompt_dummies = pd.get_dummies(
        data["prompt_id"].astype(str),
        prefix="prompt",
        drop_first=True,
        dtype=float,
    )
    controls = pd.concat(
        [
            pd.DataFrame({"candidate_length": length}),
            prompt_dummies.reset_index(drop=True),
        ],
        axis=1,
    ).to_numpy(dtype=float)

    ls_model = LinearRegression().fit(controls, lsscore)
    human_model = LinearRegression().fit(controls, human)

    ls_residual = lsscore - ls_model.predict(controls)
    human_residual = human - human_model.predict(controls)
    correlation, conventional_p = pearsonr(ls_residual, human_residual)

    return {
        "partial_correlation": float(correlation),
        "conventional_p_descriptive_only": float(conventional_p),
        "n": len(data),
        "n_prompts": data["prompt_id"].nunique(),
    }


partial_pearson = partial_correlation_controlling_length_and_prompt(
    test_df,
    rank_transform=False,
)
partial_spearman = partial_correlation_controlling_length_and_prompt(
    test_df,
    rank_transform=True,
)

length_confound_df = pd.DataFrame(
    [
        {
            "analysis": "LSScore-human zero-order Pearson",
            "correlation": safe_corr(
                test_df["ls_score"], test_human, pearsonr
            ),
            "controls": "none",
        },
        {
            "analysis": "Length-human zero-order Pearson",
            "correlation": safe_corr(
                test_df["candidate_length"], test_human, pearsonr
            ),
            "controls": "none",
        },
        {
            "analysis": "LSScore-human partial Pearson",
            "correlation": partial_pearson["partial_correlation"],
            "controls": "candidate length + prompt fixed effects",
        },
        {
            "analysis": "LSScore-human partial Spearman",
            "correlation": partial_spearman["partial_correlation"],
            "controls": "candidate length + prompt fixed effects",
        },
    ]
)

MODEL_SPECIFICATIONS = {
    "length_only": ["candidate_length"],
    "ls_score_only": ["ls_score"],
    "length_plus_ls_score": ["candidate_length", "ls_score"],
}

train_target = train_df["normalised_human_score"].to_numpy(dtype=float)
test_target = test_human
model_rows = []
heldout_predictions: dict[str, np.ndarray] = {}

for model_name, predictors in MODEL_SPECIFICATIONS.items():
    model = make_pipeline(StandardScaler(), LinearRegression())
    model.fit(
        train_df[predictors].to_numpy(dtype=float),
        train_target,
    )
    predictions = model.predict(
        test_df[predictors].to_numpy(dtype=float)
    )
    heldout_predictions[model_name] = predictions

    model_rows.append(
        {
            "model": model_name,
            "predictors": ", ".join(predictors),
            "test_rmse": float(
                np.sqrt(mean_squared_error(test_target, predictions))
            ),
            "test_mae": float(mean_absolute_error(test_target, predictions)),
            "test_r2": float(r2_score(test_target, predictions)),
            "prediction_human_pearson": safe_corr(
                predictions,
                test_target,
                pearsonr,
            ),
        }
    )

heldout_model_results_df = pd.DataFrame(model_rows)
model_index = heldout_model_results_df.set_index("model")

length_model = model_index.loc["length_only"]
joint_model = model_index.loc["length_plus_ls_score"]

incremental_value_df = pd.DataFrame(
    [
        {
            "rmse_reduction_vs_length": (
                length_model["test_rmse"] - joint_model["test_rmse"]
            ),
            "mae_reduction_vs_length": (
                length_model["test_mae"] - joint_model["test_mae"]
            ),
            "r2_increase_vs_length": (
                joint_model["test_r2"] - length_model["test_r2"]
            ),
        }
    ]
)

prediction_df = test_df[
    [
        "prompt_id",
        "ref_item_id",
        "cand_item_id",
        "normalised_human_score",
        "candidate_length",
        "ls_score",
    ]
].copy()
for model_name, predictions in heldout_predictions.items():
    prediction_df[f"prediction_{model_name}"] = predictions

length_confound_df.to_csv(
    OUTPUT_DIR / "length_partial_correlations.csv",
    index=False,
)
heldout_model_results_df.to_csv(
    OUTPUT_DIR / "length_heldout_models.csv",
    index=False,
)
incremental_value_df.to_csv(
    OUTPUT_DIR / "length_incremental_value.csv",
    index=False,
)
prediction_df.to_csv(
    OUTPUT_DIR / "length_model_test_predictions.csv",
    index=False,
)

print("\nLENGTH-CONFOUND CORRELATIONS")
display(length_confound_df.round(6))
print("\nHELD-OUT LENGTH MODEL COMPARISON")
display(heldout_model_results_df.round(6))
print("\nINCREMENTAL VALUE BEYOND LENGTH")
display(incremental_value_df.round(6))

## 14. Hierarchical prompt-cluster inference

In [ ]:
def holm_adjust(p_values: Sequence[float]) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan, dtype=float)
    valid = np.flatnonzero(np.isfinite(p_values))
    if len(valid) == 0:
        return adjusted

    ordered = valid[np.argsort(p_values[valid])]
    count = len(ordered)
    ordered_adjusted = (
        (count - np.arange(count)) * p_values[ordered]
    )
    ordered_adjusted = np.maximum.accumulate(ordered_adjusted)
    adjusted[ordered] = np.clip(ordered_adjusted, 0.0, 1.0)
    return adjusted


def add_one_two_sided_p(differences: np.ndarray) -> float:
    differences = np.asarray(differences, dtype=float)
    repetitions = len(differences)
    if repetitions == 0:
        return np.nan

    lower_tail = (np.sum(differences <= 0) + 1) / (repetitions + 1)
    upper_tail = (np.sum(differences >= 0) + 1) / (repetitions + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def hierarchical_prompt_cluster_inference(
    df: pd.DataFrame,
    baseline_columns: Sequence[str],
    *,
    repetitions: int,
    seed: int,
) -> pd.DataFrame:
    data = df.reset_index(drop=True).copy()
    prompt_values = data["prompt_id"].to_numpy()
    human = data["normalised_human_score"].to_numpy(dtype=float)
    lsscore = data["ls_score"].to_numpy(dtype=float)
    prompt_ids = data["prompt_id"].drop_duplicates().tolist()
    prompt_rows = {
        prompt_id: np.flatnonzero(prompt_values == prompt_id)
        for prompt_id in prompt_ids
    }

    observed_lsscore = average_correlation(lsscore, human)
    observed_baseline = {
        baseline: average_correlation(data[baseline], human)
        for baseline in baseline_columns
    }
    bootstrap_differences = {
        baseline: [] for baseline in baseline_columns
    }

    rng = np.random.default_rng(seed)
    for _ in tqdm(
        range(repetitions),
        desc=f"{DATASET_NAME}: hierarchical prompt bootstrap",
    ):
        sampled_prompts = rng.choice(
            prompt_ids,
            size=len(prompt_ids),
            replace=True,
        )

        sampled_blocks = []
        for prompt_id in sampled_prompts:
            original_rows = prompt_rows[prompt_id]
            sampled_rows = rng.choice(
                original_rows,
                size=len(original_rows),
                replace=True,
            )
            sampled_blocks.append(sampled_rows)

        positions = np.concatenate(sampled_blocks)
        bootstrap_human = human[positions]
        bootstrap_lsscore = average_correlation(
            lsscore[positions],
            bootstrap_human,
        )
        if not np.isfinite(bootstrap_lsscore):
            continue

        for baseline in baseline_columns:
            baseline_statistic = average_correlation(
                data[baseline].to_numpy(dtype=float)[positions],
                bootstrap_human,
            )
            if np.isfinite(baseline_statistic):
                bootstrap_differences[baseline].append(
                    bootstrap_lsscore - baseline_statistic
                )

    rows = []
    for baseline in baseline_columns:
        differences = np.asarray(
            bootstrap_differences[baseline],
            dtype=float,
        )
        if len(differences) == 0:
            raise RuntimeError(f"No valid bootstrap draws for {baseline}.")

        rows.append(
            {
                "baseline": baseline,
                "n_prompts": len(prompt_ids),
                "requested_repetitions": repetitions,
                "valid_repetitions": len(differences),
                "lsscore_observed_avgcorr": observed_lsscore,
                "baseline_observed_avgcorr": observed_baseline[baseline],
                "observed_difference": (
                    observed_lsscore - observed_baseline[baseline]
                ),
                "bootstrap_mean_difference": float(np.mean(differences)),
                "ci_2.5": float(np.quantile(differences, 0.025)),
                "ci_97.5": float(np.quantile(differences, 0.975)),
                "lsscore_wins": int(np.sum(differences > 0)),
                "bootstrap_win_rate": float(np.mean(differences > 0)),
                "bootstrap_p_two_sided": add_one_two_sided_p(differences),
            }
        )

    output = pd.DataFrame(rows)
    output["bootstrap_p_holm"] = holm_adjust(
        output["bootstrap_p_two_sided"]
    )
    output["holm_significant_0.05"] = output["bootstrap_p_holm"] < 0.05
    return output


def leave_one_prompt_out(
    df: pd.DataFrame,
    baseline_columns: Sequence[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    prompt_ids = df["prompt_id"].drop_duplicates().tolist()
    rows = []

    for omitted_prompt in prompt_ids:
        reduced = df.loc[df["prompt_id"] != omitted_prompt].copy()
        human = reduced["normalised_human_score"].to_numpy(dtype=float)
        ls_statistic = average_correlation(reduced["ls_score"], human)

        for baseline in baseline_columns:
            baseline_statistic = average_correlation(reduced[baseline], human)
            rows.append(
                {
                    "omitted_prompt": omitted_prompt,
                    "baseline": baseline,
                    "lsscore_avgcorr": ls_statistic,
                    "baseline_avgcorr": baseline_statistic,
                    "difference": ls_statistic - baseline_statistic,
                }
            )

    details = pd.DataFrame(rows)
    summary = (
        details.groupby("baseline")["difference"]
        .agg(["min", "max", "median", "count"])
        .reset_index()
        .rename(
            columns={
                "min": "lopo_min_difference",
                "max": "lopo_max_difference",
                "median": "lopo_median_difference",
                "count": "lopo_folds",
            }
        )
    )
    positive = (
        details.assign(positive=details["difference"] > 0)
        .groupby("baseline")["positive"]
        .sum()
        .reset_index(name="lopo_positive_folds")
    )
    summary = summary.merge(positive, on="baseline", validate="one_to_one")
    summary["lopo_all_positive"] = (
        summary["lopo_positive_folds"] == summary["lopo_folds"]
    )
    return details, summary


BASELINE_COLUMNS = [
    "bleu", "rouge", "bert_score", "sea_score",
    "holms", "smile_avg", "smile_hm", "hybrideval_rouge",
]
assert len(BASELINE_COLUMNS) == 8
assert "candidate_length" not in BASELINE_COLUMNS

cluster_inference_df = hierarchical_prompt_cluster_inference(
    test_df,
    BASELINE_COLUMNS,
    repetitions=BOOTSTRAP_REPETITIONS,
    seed=SEED,
)
lopo_details_df, lopo_summary_df = leave_one_prompt_out(
    test_df,
    BASELINE_COLUMNS,
)
cluster_inference_df = cluster_inference_df.merge(
    lopo_summary_df,
    on="baseline",
    how="left",
    validate="one_to_one",
)

cluster_inference_df.to_csv(
    OUTPUT_DIR / "prompt_cluster_inference.csv",
    index=False,
)
lopo_details_df.to_csv(
    OUTPUT_DIR / "leave_one_prompt_out_details.csv",
    index=False,
)

print("\nPRIMARY HIERARCHICAL PROMPT-CLUSTER INFERENCE")
display(cluster_inference_df.round(6))

for row in cluster_inference_df.itertuples():
    if row.lsscore_wins == row.valid_repetitions:
        print(
            f"LSScore won all {row.valid_repetitions:,} resamples "
            f"against {row.baseline}; add-one-corrected two-sided "
            f"p ≈ {row.bootstrap_p_two_sided:.3f}."
        )

In [ ]:
# ============================================================
# INDEPENDENT PROMPT-LEVEL SIGN-FLIP SENSITIVITY TEST
# No embedding/model inference is performed by this cell.
# Run after the existing final TEST score and bootstrap cells.
#
# This test targets the MEAN WITHIN-PROMPT AvgCorr advantage.
# This is NOT an exact test for the original pooled AvgCorr
# difference. It is a supplementary sensitivity analysis.
# Assumption: under the null, prompt-level paired differences
# are independent and their distributions are symmetric about 0.
# Prompt numbers are small; report the attainable p-value
# resolution and do NOT substitute these p-values for the
# pooled-bootstrap p-values without updating thesis estimands.
# ============================================================
import itertools
import numpy as np
import pandas as pd

AUDIT_BASELINES = [
    'bleu', 'rouge', 'bert_score', 'sea_score',
    'holms', 'smile_avg', 'smile_hm', 'hybrideval_rouge'
]
assert len(AUDIT_BASELINES) == 8
assert 'candidate_length' not in AUDIT_BASELINES
assert all(c in test_df.columns for c in AUDIT_BASELINES + [
    'prompt_id', 'normalised_human_score', 'ls_score'])

# Consistent with the AvgCorr definition in your notebooks.
def audit_avgcorr(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    pearson = pd.Series(x).corr(pd.Series(y), method='pearson')
    spearman = pd.Series(x).corr(pd.Series(y), method='spearman')
    return (pearson + spearman) / 2

prompt_ids = list(test_df['prompt_id'].drop_duplicates())
assert len(prompt_ids) >= 2

per_prompt_rows = []
for pid, group in test_df.groupby('prompt_id', sort=False):
    h = group['normalised_human_score'].to_numpy(float)
    ls = audit_avgcorr(group['ls_score'], h)
    for metric in AUDIT_BASELINES:
        other = audit_avgcorr(group[metric], h)
        per_prompt_rows.append({
            'prompt_id': pid, 'baseline': metric,
            'n_responses': len(group), 'lsscore_prompt_avgcorr': ls,
            'baseline_prompt_avgcorr': other,
            'prompt_difference': ls-other,
        })
per_prompt_audit_df = pd.DataFrame(per_prompt_rows)

# Enumerate all 2^G sign assignments; for 15 prompts this is 32,768.
# Include the observed assignment, so the null-test p-value is finite.
audit_results = []
for metric in AUDIT_BASELINES:
    sub = per_prompt_audit_df.loc[
        per_prompt_audit_df['baseline'] == metric]
    if sub['prompt_difference'].isna().any():
        raise ValueError(f'Undefined within-prompt correlation for {metric}; inspect small or constant prompts.')
    diffs = sub['prompt_difference'].to_numpy(float)
    G = len(diffs)
    if G > 20:
        raise ValueError('Too many prompts for exhaustive sign enumeration.')
    signs = np.asarray(list(itertools.product((-1., 1.), repeat=G)), float)
    null_stats = (signs * diffs).mean(axis=1)
    observed = float(diffs.mean())
    tolerance = 1e-12
    pval = float(np.mean(np.abs(null_stats) >= abs(observed) - tolerance))
    audit_results.append({
        'dataset': DATASET_NAME,
        'baseline': metric,
        'n_independent_prompts': G,
        'test_estimand': 'mean within-prompt AvgCorr difference',
        'observed_mean_prompt_difference': observed,
        'median_prompt_difference': float(np.median(diffs)),
        'prompts_lsscore_wins': int(np.sum(diffs > 0)),
        'n_exact_sign_assignments': int(2**G),
        'signflip_p_two_sided': pval,
    })

prompt_signflip_df = pd.DataFrame(audit_results)
# Holm correction over ONLY the eight LSScore-versus-baseline tests.
prompt_signflip_df['signflip_p_holm'] = holm_adjust(
    prompt_signflip_df['signflip_p_two_sided'].to_numpy(float))
prompt_signflip_df['signflip_holm_significant_0_05'] = (
    prompt_signflip_df['signflip_p_holm'] < 0.05)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
per_prompt_audit_df.to_csv(
    OUTPUT_DIR / 'prompt_signflip_per_prompt_differences.csv', index=False)
prompt_signflip_df.to_csv(
    OUTPUT_DIR / 'prompt_signflip_holm_sensitivity.csv', index=False)

print('PROMPT-LEVEL NULL SENSITIVITY (NOT POOLED AVGCORR HYPOTHESIS TEST)')
print('Independent prompts:', len(prompt_ids),
      '| smallest possible two-sided p approx:', 2 / (2 ** len(prompt_ids)))
display(prompt_signflip_df.round(6))
if 'cluster_inference_df' in globals():
    compare = cluster_inference_df[['baseline', 'observed_difference',
                                     'bootstrap_p_two_sided',
                                     'bootstrap_p_holm']].merge(
        prompt_signflip_df[['baseline', 'observed_mean_prompt_difference',
                            'signflip_p_two_sided', 'signflip_p_holm']],
        on='baseline', how='inner')
    compare.to_csv(OUTPUT_DIR / 'bootstrap_vs_prompt_signflip_audit.csv',index=False)
    display(compare.round(6))


## 15. Supplementary Williams tests

These are secondary because row-level independence is weaker than the grouped
assumption used by the prompt-cluster bootstrap.

In [ ]:
def williams_test(
    r_human_ls: float,
    r_human_baseline: float,
    r_ls_baseline: float,
    n: int,
) -> tuple[float, float]:
    if n <= 3:
        return np.nan, np.nan

    determinant = (
        1
        - r_human_ls**2
        - r_human_baseline**2
        - r_ls_baseline**2
        + 2 * r_human_ls * r_human_baseline * r_ls_baseline
    )
    denominator_term = (
        2 * determinant * (n - 1) / (n - 3)
        + ((r_human_ls + r_human_baseline) ** 2 / 4.0)
        * ((1 - r_ls_baseline) ** 3)
    )

    if denominator_term <= 0 or (1 + r_ls_baseline) <= 0:
        return np.nan, np.nan

    statistic = (
        (r_human_ls - r_human_baseline)
        * np.sqrt((n - 1) * (1 + r_ls_baseline))
        / np.sqrt(denominator_term)
    )
    p_value = 2 * stats.t.sf(abs(statistic), df=n - 3)
    return float(statistic), float(p_value)


williams_rows = []
r_human_ls = safe_corr(test_human, test_df["ls_score"], pearsonr)

for baseline in BASELINE_COLUMNS:
    r_human_baseline = safe_corr(test_human, test_df[baseline], pearsonr)
    r_ls_baseline = safe_corr(test_df["ls_score"], test_df[baseline], pearsonr)
    statistic, p_value = williams_test(
        r_human_ls,
        r_human_baseline,
        r_ls_baseline,
        len(test_df),
    )
    williams_rows.append(
        {
            "baseline": baseline,
            "williams_t": statistic,
            "williams_p_raw": p_value,
        }
    )

williams_supplement_df = pd.DataFrame(williams_rows)
williams_supplement_df["williams_p_holm"] = holm_adjust(
    williams_supplement_df["williams_p_raw"]
)
williams_supplement_df.to_csv(
    OUTPUT_DIR / "williams_tests_supplementary.csv",
    index=False,
)

print("\nSUPPLEMENTARY WILLIAMS TESTS")
display(williams_supplement_df.round(6))

## 16. Appendix-only locked-parameter ablations

In [ ]:
if RUN_ABLATION:
    ablation_specs = {
        "joint_lsscore": {
            "mode": "joint",
            "use_bp": best_params["use_bp"],
        },
        "lexical_only": {
            "mode": "lexical",
            "use_bp": best_params["use_bp"],
        },
        "semantic_recovery_only": {
            "mode": "semantic",
            "use_bp": best_params["use_bp"],
        },
        "joint_without_brevity_penalty": {
            "mode": "joint",
            "use_bp": False,
        },
    }

    ablation_rows = []
    for variant, specification in ablation_specs.items():
        train_scores = score_rows(
            train_row_data,
            best_params,
            mode=specification["mode"],
            use_bp_override=specification["use_bp"],
        )
        test_scores = score_rows(
            test_row_data,
            best_params,
            mode=specification["mode"],
            use_bp_override=specification["use_bp"],
        )

        pearson = safe_corr(test_scores, test_human, pearsonr)
        spearman = safe_corr(test_scores, test_human, spearmanr)
        kendall = safe_corr(test_scores, test_human, kendalltau)
        rmse, mae = calibrated_errors_from_arrays(
            train_scores,
            train_df["normalised_human_score"],
            test_scores,
            test_human,
        )

        temporary = test_df[["prompt_id", "normalised_human_score"]].copy()
        temporary["variant_score"] = test_scores

        ablation_rows.append(
            {
                "variant": variant,
                "pearson_r": pearson,
                "spearman_r": spearman,
                "kendall_tau": kendall,
                "avg_correlation": (pearson + spearman) / 2.0,
                "calibrated_rmse": rmse,
                "calibrated_mae": mae,
                "pairwise_accuracy": exact_pairwise_accuracy(
                    temporary,
                    "variant_score",
                ),
            }
        )

    ablation_df = pd.DataFrame(ablation_rows)
    ablation_df.to_csv(
        OUTPUT_DIR / "appendix_locked_parameter_ablations.csv",
        index=False,
    )
    print("\nAPPENDIX-ONLY LOCKED-PARAMETER ABLATIONS")
    display(ablation_df.round(6))

## 17. Final audit and manifest

In [ ]:
assert len(FULL_SEARCH_SPACE) == 3_584
assert "candidate_length" in FINAL_METRICS
assert "bootstrap_p_holm" in cluster_inference_df.columns
assert "observed_difference" in cluster_inference_df.columns
assert "lopo_min_difference" in cluster_inference_df.columns
assert test_df["ls_score"].notna().all()
assert test_df["candidate_length"].notna().all()

manifest = {
    "dataset": DATASET_NAME,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "search_specification_signature": SEARCH_SPEC_SIGNATURE,
    "data_fingerprint": DATA_FINGERPRINT,
    "run_signature": RUN_SIGNATURE,
    "search_space_size": len(FULL_SEARCH_SPACE),
    "selection_rule": SEARCH_SPECIFICATION["selection"],
    "best_params": best_params,
    "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
    "n_train": len(train_df),
    "n_validation": len(val_df),
    "n_test": len(test_df),
    "n_prompts_test": int(test_df["prompt_id"].nunique()),
    "final_metrics": FINAL_METRICS,
    "old_checkpoint_directory": str(OLD_CHECKPOINT_DIR),
    "new_checkpoint_directory": str(NEW_CHECKPOINT_DIR),
    "output_directory": str(OUTPUT_DIR),
}

with (OUTPUT_DIR / "experiment_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

save_new_checkpoint(
    "10_final_conference_analysis_complete",
    {
        "best_params": best_params,
        "results_df": results_df,
        "length_confound_df": length_confound_df,
        "heldout_model_results_df": heldout_model_results_df,
        "incremental_value_df": incremental_value_df,
        "cluster_inference_df": cluster_inference_df,
        "lopo_details_df": lopo_details_df,
        "williams_supplement_df": williams_supplement_df,
        **({"ablation_df": ablation_df} if RUN_ABLATION else {}),
    },
    metadata=manifest,
)

print("\nFinal audit passed.")
print("Dataset:", DATASET_NAME)
print("Locked parameters:", best_params)
print("Outputs:", OUTPUT_DIR)

## Supplement: TRAIN-only pseudo-reference sensitivity (locked LSScore)
Run the two following cells **after the notebook has loaded saved checkpoints and final scores**. This computes two alternative high-scoring TRAIN references per prompt, with locked parameters and TRAIN-only IDF. It will perform new MPNet inference, but no new HPO or baseline inference. Progress is checkpointed by prompt.

In [ ]:
# Run AFTER all existing notebook cells have loaded final TRAIN/TEST and best_params.
# This is a reference-selection sensitivity check, not a new HPO search.
import os, json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

REQUIRED = ['train_df', 'test_df', 'best_params', 'DATASET_NAME']
missing_globals = [name for name in REQUIRED if name not in globals()]
if missing_globals:
    raise RuntimeError('First run the main checkpoint-based notebook cells: missing ' + repr(missing_globals))
assert set(['prompt_id','cand_item_id','human_score','candidate','reference']).issubset(train_df.columns)
assert set(['prompt_id','cand_item_id','candidate','normalised_human_score','ls_score']).issubset(test_df.columns)
N_LIST = [1, 2, 3]
EMBEDDING_MODEL = 'all-mpnet-base-v2'
SENSITIVITY_DIR = Path(OUTPUT_DIR) / 'pseudo_reference_sensitivity'
SENSITIVITY_DIR.mkdir(parents=True, exist_ok=True)
# Use 2 alternative top-ranked TRAIN responses per prompt.
N_ALTERNATIVES = 2
# IDF must use original TRAIN examples, NOT validation or test examples.
train_cleaned = list(train_df['candidate'].astype(str))
primary_reference_rows = train_df[['prompt_id','reference','ref_item_id']].drop_duplicates('prompt_id')
train_cleaned += list(primary_reference_rows['reference'].astype(str))
vectorizer = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
  use_idf=True, smooth_idf=True, sublinear_tf=True, min_df=1, stop_words=None)
vectorizer.fit(train_cleaned)
idf_dict = dict(zip(vectorizer.get_feature_names_out(), vectorizer.idf_))
unseen_idf = float(np.log((1+len(train_cleaned))/1.0)+1.0)
# Existing scoring formula fast_ls_score is already defined in the main notebook.
assert 'fast_ls_score' in globals()
embedder = SentenceTransformer(EMBEDDING_MODEL)
EMBED_DIM = embedder.get_sentence_embedding_dimension()

# Build reference bank using ONLY TRAIN candidates, which exclude the primary TRAIN reference.
ALT_REFERENCE_BANK = {}
for prompt_id, group in train_df.groupby('prompt_id', sort=False):
    unique = (group[['cand_item_id','candidate','human_score']]
      .drop_duplicates('cand_item_id')
      .assign(_item_id_sort=lambda t: t['cand_item_id'].astype(str))
      .sort_values(['human_score','_item_id_sort'], ascending=[False,True]))
    if len(unique) < N_ALTERNATIVES:
        raise ValueError(f'Insufficient TRAIN alternatives for prompt {prompt_id}')
    ALT_REFERENCE_BANK[prompt_id] = [
      {'ref_rank':i+2, 'ref_item_id':row['cand_item_id'],
       'reference':str(row['candidate']), 'reference_human_score':float(row['human_score'])}
      for i,(_,row) in enumerate(unique.head(N_ALTERNATIVES).iterrows())
    ]
    print(prompt_id, 'primary:',group['ref_item_id'].iloc[0],
       '| alternatives:',[v['ref_item_id'] for v in ALT_REFERENCE_BANK[prompt_id]])
# Every alternative is a TRAIN response; no new reference uses a TEST response.
assert not (set(str(x['ref_item_id']) for alts in ALT_REFERENCE_BANK.values() for x in alts)
            & set(test_df['cand_item_id'].astype(str)))
print('TRAIN-only references selected, locked params:',best_params)

def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def ngram_idf_weights(grams, idf_dict, unseen_idf):
    if not grams:
        return np.empty(0, dtype=np.float32)

    return np.asarray([
        np.mean([idf_dict.get(tok, unseen_idf) for tok in gram])
        for gram in grams
    ], dtype=np.float32)

def encode_text_features(
    text, n_list, idf_dict, unseen_idf, batch_size=256
):
    """Embed unigram, bigram, and trigram phrases from one canonical text."""
    tokens = text.split()
    features = {}
    all_phrases, spans = [], {}

    for n in n_list:
        grams = get_ngrams(tokens, n)
        start = len(all_phrases)
        all_phrases.extend(" ".join(g) for g in grams)
        spans[n] = (start, len(all_phrases), grams)

    if all_phrases:
        all_embeddings = embedder.encode(
            all_phrases,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32, copy=False)
    else:
        all_embeddings = np.empty(
            (0, EMBED_DIM), dtype=np.float32
        )

    for n in n_list:
        start, end, grams = spans[n]
        features[n] = {
            "grams": grams,
            "emb": all_embeddings[start:end],
            "idf": ngram_idf_weights(
                grams, idf_dict, unseen_idf
            ),
        }

    features["length"] = len(tokens)
    return features

def best_cosine_both(ref_emb, cand_emb, chunk_size=512):
    """Best cosine similarity for every reference and candidate n-gram."""
    nr, nc = len(ref_emb), len(cand_emb)

    if nr == 0 or nc == 0:
        return (
            np.zeros(nr, dtype=np.float32),
            np.zeros(nc, dtype=np.float32),
        )

    best_ref = np.full(nr, -np.inf, dtype=np.float32)
    best_cand = np.full(nc, -np.inf, dtype=np.float32)

    for start in range(0, nr, chunk_size):
        end = min(start + chunk_size, nr)
        sims = ref_emb[start:end] @ cand_emb.T
        best_ref[start:end] = sims.max(axis=1)
        best_cand = np.maximum(
            best_cand, sims.max(axis=0)
        )

    return (
        np.clip(best_ref, -1, 1),
        np.clip(best_cand, -1, 1),
    )

def row_data_from_features(
    ref_features, cand_features, n_list
):
    row_data = {
        "ref_len": ref_features["length"],
        "cand_len": cand_features["length"],
        "per_n": {},
    }

    for n in n_list:
        r = ref_features[n]
        c = cand_features[n]
        ref_grams = r["grams"]
        cand_grams = c["grams"]

        if not ref_grams or not cand_grams:
            row_data["per_n"][n] = None
            continue

        cand_set = set(cand_grams)
        ref_set = set(ref_grams)
        best_sim_recall, best_sim_prec = best_cosine_both(
            r["emb"], c["emb"]
        )

        row_data["per_n"][n] = {
            "exact_recall": np.asarray(
                [g in cand_set for g in ref_grams],
                dtype=bool,
            ),
            "best_sim_recall": best_sim_recall,
            "idf_recall": r["idf"],
            "exact_prec": np.asarray(
                [g in ref_set for g in cand_grams],
                dtype=bool,
            ),
            "best_sim_prec": best_sim_prec,
            "idf_prec": c["idf"],
        }

    return row_data

In [ ]:
# Resume-safe scoring: one TEST candidate is encoded once for BOTH alternative references.
# Each prompt's completed output is cached as a CSV, independent of other prompts.
params = best_params
for prompt_id, group in tqdm(test_df.groupby('prompt_id',sort=False), total=test_df['prompt_id'].nunique(), desc='Prompts'):
    safe_pid = str(prompt_id).replace('/','_').replace('\\','_')
    output_path = SENSITIVITY_DIR / f'prompt_{safe_pid}.csv'
    if output_path.exists():
        previous = pd.read_csv(output_path)
        if (len(previous)==len(group) and
           set(previous['cand_item_id'].astype(str))==set(group['cand_item_id'].astype(str)) and
           all(f'ls_ref_{rank}' in previous.columns for rank in (2,3))):
            print(f'Skipping checkpointed prompt {prompt_id}')
            continue
        raise RuntimeError(f'Existing checkpoint mismatch: {output_path}; do not overwrite blindly')
    alt_refs = ALT_REFERENCE_BANK[prompt_id]
    ref_features = {alt['ref_rank']:encode_text_features(alt['reference'],N_LIST,idf_dict,unseen_idf)
                   for alt in alt_refs}
    output_rows=[]
    for _, row in tqdm(group.iterrows(),total=len(group),leave=False,desc=f'{prompt_id}'):
        candidate_feat = encode_text_features(str(row['candidate']),N_LIST,idf_dict,unseen_idf)
        entry = {'prompt_id':prompt_id,'cand_item_id':row['cand_item_id'],
                 'human':float(row['normalised_human_score']),
                 'ls_ref_1':float(row['ls_score'])}
        for alt in alt_refs:
            rank=alt['ref_rank']
            features = row_data_from_features(ref_features[rank],candidate_feat,N_LIST)
            entry[f'ls_ref_{rank}'] = float(fast_ls_score(features,N_LIST,
                 params['alpha'],params['tau'],params['delta'],params['beta'],use_bp=params['use_bp']))
        output_rows.append(entry)
    temporary_path = output_path.with_suffix('.csv.tmp')
    pd.DataFrame(output_rows).to_csv(temporary_path,index=False)
    os.replace(temporary_path,output_path)
    print('Saved:',output_path)

all_chunks=[]
for pid, group in test_df.groupby('prompt_id',sort=False):
    safe_pid = str(pid).replace('/','_').replace('\\','_')
    path=SENSITIVITY_DIR / f'prompt_{safe_pid}.csv'
    if not path.exists():raise RuntimeError(f'Missing saved prompt: {path}')
    all_chunks.append(pd.read_csv(path))
combined=pd.concat(all_chunks,ignore_index=True)
assert len(combined)==len(test_df)
assert all(combined[c].notna().all() for c in ['ls_ref_1','ls_ref_2','ls_ref_3','human'])
combined.to_csv(SENSITIVITY_DIR/'all_test_alternative_reference_scores.csv',index=False)

def avgcorr(sub, col):
    x=sub[col].to_numpy(dtype=float)
    y=sub['human'].to_numpy(dtype=float)
    if len(x)<3 or np.std(x)==0 or np.std(y)==0:return float('nan')
    return float((pearsonr(x,y).statistic + spearmanr(x,y).statistic)/2)

primary=avgcorr(combined,'ls_ref_1')
rows=[]
for rank in (1,2,3):
    col=f'ls_ref_{rank}'
    r_pearson=float(pearsonr(combined[col],combined['human']).statistic)
    r_spearman=float(spearmanr(combined[col],combined['human']).statistic)
    rank_agreement=float(spearmanr(combined['ls_ref_1'],combined[col]).statistic)
    rows.append({'dataset':DATASET_NAME,'reference_rank':rank,'n_test':len(combined),
        'pearson':r_pearson,'spearman':r_spearman,'avgcorr':(r_pearson+r_spearman)/2,
        'delta_avgcorr_vs_primary':(r_pearson+r_spearman)/2-primary,
        'candidate_ranking_spearman_vs_primary':rank_agreement})
summary=pd.DataFrame(rows)
perprompt=[]
for pid,g in combined.groupby('prompt_id',sort=False):
    base=avgcorr(g,'ls_ref_1')
    for rank in (1,2,3):
        ac=avgcorr(g,f'ls_ref_{rank}')
        perprompt.append({'dataset':DATASET_NAME,'prompt_id':pid,'reference_rank':rank,
              'n_test':len(g),'avgcorr':ac,'delta_from_primary':ac-base})
perprompt=pd.DataFrame(perprompt)
summary.to_csv(SENSITIVITY_DIR/'reference_sensitivity_summary.csv',index=False)
perprompt.to_csv(SENSITIVITY_DIR/'reference_sensitivity_per_prompt.csv',index=False)
with open(SENSITIVITY_DIR/'reference_bank.json','w') as f:
    json.dump({str(k):v for k,v in ALT_REFERENCE_BANK.items()},f,indent=2,default=str)
print('\nPSEUDO-REFERENCE SENSITIVITY: FULL TEST')
display(summary)
print('\nPSEUDO-REFERENCE SENSITIVITY: WITHIN PROMPT')
display(perprompt)
print('Outputs:',SENSITIVITY_DIR)
print('Note: This experiment tests LSScore reference-choice sensitivity only; it does not recompute the eight baselines with alternative references.')
